In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, max_error
from sklearn.preprocessing import StandardScaler

# Load the dataset
data = pd.read_csv("data\Temp_Asu20092021.csv")

# Drop rows with missing values
data.dropna(inplace=True)

# Convert 'Fecha' column to datetime type
data['Fecha'] = pd.to_datetime(data['Fecha'])

# Set 'Fecha' column as index
data.set_index('Fecha', inplace=True)

# Upsample to hourly intervals and interpolate missing values
data_interpolated = data.resample('1H').interpolate(method='spline', order=3)




In [3]:

data = data[data.index.year > 2013]

serie=data['Temperatura24h']

KeyError: 'Temperatura24h'

In [ ]:
## The function below transforms the input series and window-size into a set of input/output pairs for our RNN model
def window_step_transform_series(series,window_size,step, pred_horizon):
    # containers for input/output pairs
    X = []
    y = []
    
    for i in range(window_size, len(series),step):
        X.append(series[i - window_size:i])
        y.append(series[i:(i+pred_horizon)])
        
    # reshape each 
    X = np.asarray(X)
    X.shape = (np.shape(X)[0:2])
    y = np.asarray(y)
    y.shape = (np.shape(y)[0:2])
    
    return X,y

In [ ]:
X,y = window_step_transform_series(series = serie,window_size = 24*7,step=24,pred_horizon=24)

In [ ]:

# Define cross-validation strategy
train_indices = (data.index.year < 2016)  # Training data until 2015
val_indices = ((data.index.year >= 2016) & (data.index.year <= 2018))  # Validation data from 2016 to 2018
finaltrain_indices = (data.index.year < 2019)  # Training data until 2019
finaltest_indices = ((data.index.year >= 2019) | (data.index.year >= 2021))  # Test data from 2019 and 2021



In [ ]:

# Define parameter grid for grid search
hidden_layer_sizes = [(50,), (100,), (200,)]

# Initialize lists to store training and validation metrics for each hidden size
train_metrics = []
val_metrics = []

# Perform grid search using for loops
for hidden_size in hidden_layer_sizes:
    print(hidden_size)
    # Initialize lists to store evaluation metrics for each fold
    fold_train_rmse = []
    fold_train_mape = []
    fold_train_max_err = []
    fold_val_rmse = []
    fold_val_mape = []
    fold_val_max_err = []
    
    # Perform manual cross-validation in 2016, 2017, 2018
    for year in range(2015, 2019):
        print(year)
        # Define train and validation indices for the current year
        train_indices = (data.index.year < year)  # Training data until the current year
        val_indices = ((data.index.year >= year) & (data.index.year < year + 1))  # Validation data for the current year

        # Split data into training and validation sets
        X_train, X_val = X[train_indices], X[val_indices]
        y_train, y_val = y[train_indices], y[val_indices]

        # Initialize and compile TensorFlow model
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden_size[0], activation='relu'),
            tf.keras.layers.Dense(24)
        ])
        model.compile(optimizer='adam', loss='mse')

        # Create StandardScaler instance
        scaler = StandardScaler()

        # Scale training data
        X_train_scaled = scaler.fit_transform(X_train)

        # Train the model
        model.fit(X_train_scaled, y_train, epochs=50, verbose=0)

        # Make predictions on training set
        y_train_pred = model.predict(X_train_scaled).flatten()

        # Compute evaluation metrics on training set
        rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
        mape_train = np.mean(np.abs((y_train - y_train_pred) / y_train)) * 100
        max_err_train = max_error(y_train, y_train_pred)
        print(rmse_train)
        # Append metrics to lists
        fold_train_rmse.append(rmse_train)
        fold_train_mape.append(mape_train)
        fold_train_max_err.append(max_err_train)

        # Scale validation data
        X_val_scaled = scaler.transform(X_val)

        # Make predictions on validation set
        y_val_pred = model.predict(X_val_scaled).flatten()

        # Compute evaluation metrics on validation set
        rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
        mape_val = np.mean(np.abs((y_val - y_val_pred) / y_val)) * 100
        max_err_val = max_error(y_val, y_val_pred)

        # Append metrics to lists
        fold_val_rmse.append(rmse_val)
        fold_val_mape.append(mape_val)
        fold_val_max_err.append(max_err_val)

    # Compute average metrics for the fold
    avg_train_rmse = np.mean(fold_train_rmse)
    avg_train_mape = np.mean(fold_train_mape)
    avg_train_max_err = np.mean(fold_train_max_err)
    avg_val_rmse = np.mean(fold_val_rmse)
    avg_val_mape = np.mean(fold_val_mape)
    avg_val_max_err = np.mean(fold_val_max_err)

    # Store average metrics
    train_metrics.append((hidden_size, avg_train_rmse, avg_train_mape, avg_train_max_err))
    val_metrics.append((hidden_size, avg_val_rmse, avg_val_mape, avg_val_max_err))




In [ ]:
# Display training and validation metrics
print("Training Metrics:")
print("Hidden Size\tRMSE\tMAPE\tMax Error")
for metrics in train_metrics:
    print(f"{metrics[0]}\t{metrics[1]:.4f}\t{metrics[2]:.4f}\t{metrics[3]:.4f}")

print("\nValidation Metrics:")
print("Hidden Size\tRMSE\tMAPE\tMax Error")
for metrics in val_metrics:
    print(f"{metrics[0]}\t{metrics[1]:.4f}\t{metrics[2]:.4f}\t{metrics[3]:.4f}")

# Find the hidden size with the lowest average validation RMSE
best_hidden_size = None
best_avg_val_rmse = float('inf')

for hidden_size, avg_val_rmse, _, _ in val_metrics:
    if avg_val_rmse < best_avg_val_rmse:
        best_hidden_size = hidden_size
        best_avg_val_rmse = avg_val_rmse

print("Best Hidden Size:", best_hidden_size)

# Initialize the best model with the selected hidden size
best_model = tf.keras.Sequential([
    tf.keras.layers.Dense(best_hidden_size[0], activation='relu'),
    tf.keras.layers.Dense(1)
])
best_model.compile(optimizer='adam', loss='mse')

In [ ]:

# Create StandardScaler instance
scaler = StandardScaler()
# Scale final training data
X_final_train = X[finaltrain_indices]
y_final_train = y[finaltrain_indices]
X_final_train_scaled = scaler.fit_transform(X_final_train)

# Train the best model on the final training data
best_model.fit(X_final_train_scaled, y_final_train, epochs=50, verbose=0)